In [1]:
%reset -f
import numpy as np
import pandas as pd
from pathlib import Path
import torch

import sys
sys.path.append("../../tools")
import prepare as prep
import functions as func

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [2]:
import importlib
#importlib.reload(acht)


In [3]:
load_dir1 = Path(f"../../saved_models/sta_entropy_15fold/sp3to20_MD_transpt3_r9")
load_dir2 = Path(f"../../saved_models/diversity_15fold/sp3to20_MD_transpt3_r1")
load_dir3 = Path(f"../../saved_models/num_stable_states_15fold/sp3to20_NS_transpt3_r7")
load_dir4 = Path(f"../../saved_models/fluctuation_15fold_v2/sp3to20_Fl_transpt3_r1")
load_dir5 = Path(f"../../saved_models/swi_species_15fold/sp3to20_SS_transpt3_r13")
emodel = func.load_saved_model_MD(load_dir1, device)
dmodel = func.load_saved_model_MD(load_dir2, device)
nmodel = func.load_saved_model_NS(load_dir3, device)
fmodel = func.load_saved_model_MD(load_dir4, device)
smodel = func.load_saved_model_SS(load_dir5, device)
epredictor = func.Predictor(emodel)
dpredictor = func.Predictor(dmodel)
npredictor = func.Predictor(nmodel)
fpredictor = func.Predictor(fmodel)
spredictor = func.Predictor(smodel)


c:\Users\HW\anaconda3\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [ ]:
# Fix mu, D, randomize richness and gamma under different density and positive proportion conditions.
save_path = "random2.csv"

datasize = 8000
max_species = 20
all_data = []

for NumSpecies in range(3, max_species + 1):

    featgam = NumSpecies**2 - NumSpecies
    total_dim = NumSpecies + featgam + NumSpecies
    density = np.zeros(datasize)
    pos_inall = np.zeros(datasize)

    Xnum = np.zeros((datasize, total_dim))

    Xnum[:, 0:NumSpecies] = 0.1 + 0.9 * np.random.rand(datasize, NumSpecies)

    for i in range(datasize):

        neg_fraq=0.5+0.5*np.random.rand()
        pos_fraq=np.random.rand()*(1-neg_fraq)

        m_neg = 5*np.random.rand()
        m_pos = 5*np.random.rand()
        gamma_sigma = 0.2
        gammas = []

        for m in range(NumSpecies):
            for n in range(NumSpecies):
                if m != n:
                    r = np.random.rand()
                    if r < neg_fraq:
                        #gammas.append(-m_neg * ((1 - gamma_sigma) + 2 * gamma_sigma * np.random.rand()))
                        gammas.append(-m_neg*(0.5+np.random.rand()))
                    elif r > 1 - pos_fraq:
                        #gammas.append(m_pos * ((1 - gamma_sigma) + 2 * gamma_sigma * np.random.rand()))
                        gammas.append(m_pos*(0.5+np.random.rand())) 
                    else:
                        gammas.append(0.0)

        gammas = np.array(gammas)
        density[i] = np.mean(gammas != 0)
        n_pos = np.sum(gammas > 0)
        n_nonzero = np.sum(gammas != 0)
        if n_nonzero > 0:
            pos_inall[i] = n_pos / n_nonzero
        else:
            pos_inall[i] = np.nan
        Xnum[i, NumSpecies:(NumSpecies + featgam)] = gammas

    Xnum[:, (NumSpecies + featgam):(NumSpecies + featgam + NumSpecies)] = \
        1e-06 / NumSpecies * np.random.rand(datasize, NumSpecies)

    # ===== padding =====
    #Xnum_pad = prep.pad_Xnum_to_maxspecies(Xnum, NumSpecies, max_species)
    Xnum_pad = prep.pad_Xnum_to_maxspecies(Xnum, NumSpecies, 20)

    # ===== Add new statistics =====
    richness_col = np.full(datasize, NumSpecies)
    den_arr = np.full(datasize, density)
    pos_arr = np.full(datasize, pos_inall)

    gamma_flatten = Xnum[:, NumSpecies:(NumSpecies + featgam)]
    gamma_neg_arr = np.array([
        np.mean(np.abs(row[row < 0])) if np.any(row < 0) else 0
        for row in gamma_flatten])
    gamma_pos_arr = np.array([
        np.mean(np.abs(row[row > 0])) if np.any(row > 0) else 0
        for row in gamma_flatten])  
    gamma_mean_arr = np.array([
        np.mean(np.abs(row[row != 0])) if np.any(row != 0) else 0
        for row in gamma_flatten])  
    
    relative_entropy = epredictor.predict_array_MD(Xnum_pad, device)
    entropy = relative_entropy*(NumSpecies-1)+1
    relative_diversity = dpredictor.predict_array_MD(Xnum_pad, device)
    diversity = relative_diversity*(NumSpecies-1)+1
    num_stable_states = npredictor.predict_array_NS(Xnum_pad, device)
    fluctuation = fpredictor.predict_array_Fl(Xnum_pad, device)
    swi_species = spredictor.predict_array_SS(Xnum_pad, device)
    
    extra = np.column_stack([richness_col, den_arr, pos_arr, gamma_neg_arr, gamma_pos_arr, gamma_mean_arr,
                                relative_entropy, entropy, relative_diversity, diversity, num_stable_states,
                                fluctuation, swi_species])

    # ===== Concatenate =====
    X_full = np.concatenate([Xnum_pad, extra], axis=1)
    all_data.append(X_full)

    print(f"Processed NumSpecies = {NumSpecies}")
    

Processed NumSpecies = 3
Processed NumSpecies = 4
Processed NumSpecies = 5
Processed NumSpecies = 6
Processed NumSpecies = 7
Processed NumSpecies = 8
Processed NumSpecies = 9
Processed NumSpecies = 10
Processed NumSpecies = 11
Processed NumSpecies = 12
Processed NumSpecies = 13
Processed NumSpecies = 14
Processed NumSpecies = 15
Processed NumSpecies = 16
Processed NumSpecies = 17
Processed NumSpecies = 18
Processed NumSpecies = 19
Processed NumSpecies = 20


In [5]:
# ===== Merge all =====
all_data = np.vstack(all_data)

df = pd.DataFrame(all_data)
n_feature_cols = max_species**2+max_species
df.columns = list(df.columns[:n_feature_cols]) + ["richness", "density", "pos_inall", "gamma_neg_mean", "gamma_pos_mean", 
                                                  "gamma_mean", "relative_H", "H", "relative_D", "D", "states_number", 
                                                  "fluctuation"]+ [f"switching_sp{i+1}" for i in range(max_species)]
df.to_csv(save_path, index=False)

print(f"\ndf size = {df.shape}")
print(f"\nSaved ALL data to: {save_path}")



df size = (144000, 452)

Saved ALL data to: random2.csv
